In [1]:
import pandas as pd
import re

In [2]:
df_b1 = pd.read_csv("../data/processed/bioshock_1_clean.csv", parse_dates=["review_date"])
df_b2 = pd.read_csv("../data/processed/bioshock_2_clean.csv", parse_dates=["review_date"])
df_inf = pd.read_csv("../data/processed/bioshock_infinite_clean.csv", parse_dates=["review_date"])

In [3]:
THEMES = {
    "narrative":["twist", "would you kindly", "betrayal", "obedience", "protector", "flashback", "redemption", "timeline", "multiverse", "realities", "constants", "genes", "genetic engineering", "gene altering", "autonomy", "agency", "conditioned", "kidnapping", "storytelling", "narrative", "writing", "dialogue", "story", "plot", "ending"],
    "setting" :["rapture", "underwater", "city", "art deco", "retro", "theatre", "columbia", "floating", "sky", "americana", "steampunk", "utopia", "environment", "architecture", "dystopian", "ocean", "ruins", "aesthetic", "lighthouse", "pavilion", "arcadia", "prometheus", "amusements"],
    "atmosphere" :["horror", "eerie", "scary", "tense", "immersive", "uncomfortable", "disturbing", "mystery", "dread", "claustrophobic", "twisted", "unsettling"],
    "characters" :["jack", "ryan", "atlas", "fontaine", "tenenbaum", "cohen", "little sister", "daddies", "daddy", "splicer", "delta", "eleanor", "lamb", "sofia", "sinclair", "grace", "big sister", "poole", "booker", "elizabeth", "comstock", "songbird", "twins", "daisy", "turrets", "mosquito", "zeppelin", "barrage", "handyman", "fireman", "zealot", "motorized patriot", "siren", "boy of silence", "lutece", "suchong"],
    "combat" :["plasmids", "combat", "movement", "quantum tear", "weapons", "gunplay", "shooting", "vigors", "guns", "wrench", "research camera", "drill", "rivet", "hack", "sky-hook", "adam", "eve", "tonics", "difficulty"],
    "ideology" :["capitalism", "greed", "individualism", "free will", "wealth disparity", "exploitation", "slavery", "selfishness", "political", "religious", "religion", "oppression", "collectivism", "utilitarianism", "exceptionalism", "communism", "police state", "ayn rand", "authoritarian", "right wing", "morality", "objectivism", "racism", "nationalism"],
    "visual_audio" :["lighting", "audio", "graphics", "music", "voice acting", "sound design", "style", "soundtrack"],
    "technical" :["crash", "stutter", "performance", "port", "patch", "fov", "mouse", "bugs", "optimization", "fps", "controls", "resolution", "freezing", "freeze", "glitching", "glitch", "glitches", "crashes", "crashed", "crashing", "laggy"],
    "pacing" :["repetitive", "short", "long", "boring", "tedious", "replayability", "backtrack", "drags", "filler", "padding"],
}

In [4]:
def detect_themes(text, themes=THEMES):
    text = str(text).lower()
    return {name: any(re.search(rf"\b{re.escape(kw)}\b", text) for kw in keywords)
            for name, keywords in themes.items()}

In [5]:
detect_themes("The story twist in Rapture blew my mind, but it crashes constantly") # example review

{'narrative': True,
 'setting': True,
 'atmosphere': False,
 'characters': False,
 'combat': False,
 'ideology': False,
 'visual_audio': False,
 'technical': True,
 'pacing': False}

In [6]:
for frame in (df_b1, df_b2, df_inf):
    theme_flags = frame["review"].apply(detect_themes).apply(pd.Series)
    frame[theme_flags.columns] = theme_flags

In [7]:
theme_cols = list(THEMES.keys())

salience = pd.DataFrame({
    "BioShock 1": df_b1[theme_cols].mean(),
    "BioShock 2": df_b2[theme_cols].mean(),
    "Infinite": df_inf[theme_cols].mean(),
})
(salience * 100).round(1)

,BioShock 1,BioShock 2,Infinite
narrative,24.6,23.5,40.4
setting,8.6,8.3,9.3
atmosphere,4.9,2.6,3.0
characters,6.0,12.5,9.3
combat,10.0,13.2,13.2
ideology,1.3,1.0,1.9
visual_audio,11.1,8.2,12.1
technical,25.0,40.3,7.7
pacing,4.9,5.3,7.5


In [8]:
def salience_by_verdict(frame, theme_cols):
    return (frame.groupby("voted_up")[theme_cols].mean().T * 100).round(1)

salience_by_verdict(df_b1, theme_cols)

voted_up,False,True
narrative,10.6,27.6
setting,5.6,9.3
atmosphere,1.9,5.6
characters,5.2,6.2
combat,8.0,10.5
ideology,0.6,1.4
visual_audio,13.8,10.5
technical,60.5,17.2
pacing,6.4,4.5


In [9]:
def salience_by_verdict(frame, theme_cols):
    return (frame.groupby("voted_up")[theme_cols].mean().T * 100).round(1)

salience_by_verdict(df_b2, theme_cols)

voted_up,False,True
narrative,11.9,28.3
setting,3.7,10.2
atmosphere,1.1,3.2
characters,7.0,14.8
combat,7.9,15.4
ideology,0.4,1.2
visual_audio,7.9,8.3
technical,75.3,26.0
pacing,5.0,5.5


In [10]:
def salience_by_verdict(frame, theme_cols):
    return (frame.groupby("voted_up")[theme_cols].mean().T * 100).round(1)

salience_by_verdict(df_inf, theme_cols)

voted_up,False,True
narrative,38.4,40.6
setting,12.6,9.0
atmosphere,3.7,3.0
characters,12.9,9.0
combat,23.6,12.2
ideology,6.0,1.5
visual_audio,11.7,12.2
technical,12.0,7.3
pacing,19.0,6.3


In [11]:
def salience_by_era(frame, theme_cols):
    return (frame.groupby("era")[theme_cols].mean().T * 100).round(1)

In [12]:
salience_by_era(df_b1, theme_cols)

era,later,launch
narrative,25.7,16.1
setting,8.6,9.0
atmosphere,5.3,2.4
characters,6.0,5.6
combat,10.1,9.6
ideology,1.4,0.5
visual_audio,9.4,23.1
technical,19.5,63.9
pacing,4.8,5.3


In [13]:
salience_by_era(df_b2, theme_cols)

era,later,launch
narrative,24.1,17.7
setting,8.4,7.0
atmosphere,2.7,1.2
characters,12.7,9.9
combat,13.5,10.2
ideology,1.1,0.3
visual_audio,7.5,15.5
technical,37.9,66.2
pacing,5.4,4.3


In [14]:
salience_by_era(df_inf, theme_cols)

era,later,launch
narrative,39.8,43.9
setting,9.2,10.1
atmosphere,3.0,3.4
characters,9.4,8.9
combat,12.9,15.3
ideology,1.9,1.8
visual_audio,11.7,14.4
technical,7.8,7.2
pacing,7.2,8.9


In [15]:
for name, frame in [("BioShock 1", df_b1), ("BioShock 2", df_b2), ("Infinite", df_inf)]:
    print(name)
    print((frame.groupby("era")["voted_up"].mean() * 100).round(1))

BioShock 1
era
later     86.4
launch    50.8
Name: voted_up, dtype: float64
BioShock 2
era
later     73.6
launch    42.5
Name: voted_up, dtype: float64
Infinite
era
later     90.5
launch    94.9
Name: voted_up, dtype: float64


In [16]:
(df_b1[["author.playtime_forever", "author.playtime_at_review"]] / 60).describe().round(1)

,author.playtime_forever,author.playtime_at_review
count,26155.0,26116.0
mean,23.1,14.9
std,242.5,101.6
min,0.1,0.0
25%,9.2,5.0
50%,14.9,10.8
75%,23.9,17.5
max,34226.2,16040.7


In [17]:
def add_playtime_segment(frame):
    hours = frame["author.playtime_at_review"] / 60
    frame["segment"] = pd.cut(hours, bins=[0, 2, 12, float("inf")],
                              labels=["brief (<2h)", "moderate (2-12h)", "extended (12h+)"])

In [18]:
for frame in (df_b1, df_b2, df_inf):
    add_playtime_segment(frame)

In [19]:
df_b1["segment"].value_counts()

segment
moderate (2-12h)    12098
extended (12h+)     11523
brief (<2h)          2495
Name: count, dtype: int64

In [20]:
(df_b1.groupby("segment", observed=True)["voted_up"].mean() * 100).round(1)

segment
brief (<2h)         42.6
moderate (2-12h)    81.9
extended (12h+)     90.9
Name: voted_up, dtype: float64

In [21]:
(df_b1.groupby("segment", observed=True)[theme_cols].mean().T * 100).round(1)

segment,brief (<2h),moderate (2-12h),extended (12h+)
narrative,10.0,19.9,32.6
setting,6.6,6.5,11.2
atmosphere,2.7,4.0,6.4
characters,3.3,5.5,7.1
combat,4.9,7.6,13.7
ideology,0.6,0.8,1.9
visual_audio,14.5,8.7,12.8
technical,41.8,22.6,23.7
pacing,3.7,4.0,6.1


In [22]:
df_b2["segment"].value_counts()

segment
moderate (2-12h)    6077
extended (12h+)     4853
brief (<2h)         1071
Name: count, dtype: int64

In [23]:
(df_b2.groupby("segment", observed=True)["voted_up"].mean() * 100).round(1)

segment
brief (<2h)         34.5
moderate (2-12h)    69.4
extended (12h+)     81.1
Name: voted_up, dtype: float64

In [24]:
(df_b2.groupby("segment", observed=True)[theme_cols].mean().T * 100).round(1)

segment,brief (<2h),moderate (2-12h),extended (12h+)
narrative,10.0,19.3,31.8
setting,4.3,5.8,12.3
atmosphere,1.4,2.0,3.6
characters,5.6,10.6,16.4
combat,5.2,10.4,18.6
ideology,0.6,0.6,1.6
visual_audio,9.0,6.3,10.4
technical,48.5,40.8,37.8
pacing,3.3,4.2,7.2


In [25]:
df_inf["segment"].value_counts()

segment
extended (12h+)     27085
moderate (2-12h)    18371
brief (<2h)          1683
Name: count, dtype: int64

In [26]:
(df_inf.groupby("segment", observed=True)["voted_up"].mean() * 100).round(1)

segment
brief (<2h)         74.9
moderate (2-12h)    91.1
extended (12h+)     92.2
Name: voted_up, dtype: float64

In [27]:
(df_inf.groupby("segment", observed=True)[theme_cols].mean().T * 100).round(1)

segment,brief (<2h),moderate (2-12h),extended (12h+)
narrative,23.3,36.1,44.4
setting,7.0,7.1,11.0
atmosphere,2.3,2.3,3.6
characters,5.3,7.8,10.6
combat,6.8,11.1,15.1
ideology,1.5,1.4,2.3
visual_audio,10.6,11.3,12.8
technical,9.2,6.9,8.2
pacing,5.6,6.9,8.0


In [28]:
themes = ["atmosphere", "combat", "pacing"]
for name, df_g in [("bioshock_1", df_b1), ("bioshock_2", df_b2)]:
    for verdict, label in [(True, "positive"), (False, "negative")]:
        sub = df_g[df_g["voted_up"] == verdict]
        for t in themes:
            pct = sub[t].mean() * 100
            print(f"{name} {label} n={len(sub)}: {t} {pct:.1f}%")

bioshock_1 positive n=21457: atmosphere 5.6%
bioshock_1 positive n=21457: combat 10.5%
bioshock_1 positive n=21457: pacing 4.5%
bioshock_1 negative n=4698: atmosphere 1.9%
bioshock_1 negative n=4698: combat 8.0%
bioshock_1 negative n=4698: pacing 6.4%
bioshock_2 positive n=8528: atmosphere 3.2%
bioshock_2 positive n=8528: combat 15.4%
bioshock_2 positive n=8528: pacing 5.5%
bioshock_2 negative n=3480: atmosphere 1.1%
bioshock_2 negative n=3480: combat 7.9%
bioshock_2 negative n=3480: pacing 5.0%


In [29]:
df_inf["voted_up"].value_counts()

voted_up
True     43050
False     4182
Name: count, dtype: int64

In [30]:
df_b1["voted_up"].value_counts()

voted_up
True     21457
False     4698
Name: count, dtype: int64

In [31]:
df_b2["voted_up"].value_counts()

voted_up
True     8528
False    3480
Name: count, dtype: int64

In [33]:
df_b1.to_csv("../data/processed/bioshock_1_themed.csv", index=False)
df_b2.to_csv("../data/processed/bioshock_2_themed.csv", index=False)
df_inf.to_csv("../data/processed/bioshock_infinite_themed.csv", index=False)